In [1]:
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
with open('dataset/processed/corpus_clean.txt', 'r') as f:
    text = f.read()

characters = sorted(list(set(text)))
vocab_size = len(characters)

char_to_idx = { ch:i for i,ch in enumerate(characters) }
idx_to_char = { i:ch for i,ch in enumerate(characters) }
encode = lambda xs: [char_to_idx[x] for x in xs]
decode = lambda xs: ''.join([idx_to_char[x] for x in xs])

In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

def get_batch(split, batch_size, context_size):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_size, (batch_size,))
    x = torch.stack([data[i:i+context_size] for i in ix])
    y = torch.stack([data[i+1:i+context_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [5]:
def generate(model, context_size, start_idx, number_of_tokens):
    idx = start_idx
    for _ in range(number_of_tokens):
        idx_cond = idx[:, -context_size:]
        logits, loss = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [6]:
@torch.no_grad()
def estimate_loss(model, batch_size, context_size, eval_iters=100):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, context_size)
            logits, loss = model(X, Y)
            losses[k] = loss.mean().item()
        out[split] = losses.mean()
    model.train()
    return out


def train(model, steps, batch_size, context_size, report_frequency=1000, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for step in range(steps):
        xb, yb = get_batch('train', batch_size, context_size)

        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.mean().backward()
        optimizer.step()

        if step % report_frequency == 0 or step == steps - 1:
            losses = estimate_loss(model, batch_size, context_size)
            print(f'step {step}  train loss: {losses["train"]:.4f}  val loss: {losses["val"]:.4f}')

In [7]:
def train_generate_print(model, steps=5000, batch_size=32, context_size=8, lr=1e-3):
    train(model, steps, batch_size, context_size, lr=lr)

    start_idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(
        generate(model, context_size, start_idx=start_idx, number_of_tokens=300)[0].tolist()
    ))

In [8]:
# gpt2:        n_layer=12, n_head=12, n_embd=768   # 124M params
# gpt2-medium: n_layer=24, n_head=16, n_embd=1024  # 350M params
# gpt2-large:  n_layer=36, n_head=20, n_embd=1280  # 774M params

batch_size   = 64
context_size = 256
lr           = 3e-4
n_embd       = 384
n_heads      = 6
n_layer      = 6

In [9]:
from gpt import GPT

m = GPT(
    vocab_size,
    n_embd=n_embd,
    context_size=context_size,
    n_head=n_heads,
    n_layer=n_layer
).to(device)

if torch.cuda.device_count() > 1:
    print(f'using {torch.cuda.device_count()} GPUs')
    m = nn.DataParallel(m)

print(f'params: {sum(p.numel() for p in m.parameters()) / 1e6:.2f}M')

train(m, steps=25000, batch_size=batch_size, context_size=context_size, lr=lr)

start_idx = torch.zeros((1, 1), dtype=torch.long, device=device)
base_model = m.module if isinstance(m, nn.DataParallel) else m
with torch.no_grad():
    idx = start_idx
    for _ in range(300):
        idx_cond = idx[:, -context_size:]
        logits, _ = base_model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
print(decode(idx[0].tolist()))

using 2 GPUs
params: 10.81M


/home/har5ha/Desktop/BuildFellowship/My Project/venv/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


step 0  train loss: 3.8476  val loss: 3.8355
step 1000  train loss: 1.5969  val loss: 1.6640
step 2000  train loss: 1.3110  val loss: 1.4172
step 3000  train loss: 1.2014  val loss: 1.3230
step 4000  train loss: 1.1309  val loss: 1.2714
step 5000  train loss: 1.0857  val loss: 1.2342
step 6000  train loss: 1.0482  val loss: 1.2031
step 7000  train loss: 1.0231  val loss: 1.2045
step 8000  train loss: 0.9962  val loss: 1.1945
step 9000  train loss: 0.9771  val loss: 1.1858
step 10000  train loss: 0.9572  val loss: 1.1868
step 11000  train loss: 0.9339  val loss: 1.1772
step 12000  train loss: 0.9175  val loss: 1.1737
step 13000  train loss: 0.9007  val loss: 1.1768
step 14000  train loss: 0.8828  val loss: 1.1821
step 15000  train loss: 0.8726  val loss: 1.1835
step 16000  train loss: 0.8581  val loss: 1.1857
step 17000  train loss: 0.8425  val loss: 1.1863
step 18000  train loss: 0.8336  val loss: 1.2001
step 19000  train loss: 0.8177  val loss: 1.2083
step 20000  train loss: 0.8032  v

In [10]:
state = m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()
torch.save(state, 'luffy_gpt.pth')
print('saved → luffy_gpt.pth')

saved → luffy_gpt.pth


# BPE tokenizer training

In [11]:
import sys
sys.path.append('tokenizer')
from tokenizer import LuffyTokenizer

tok = LuffyTokenizer()
print(tok)

# compare vs char-level
with open('dataset/processed/corpus_clean.txt', 'r') as f:
    raw = f.read()

bpe_tokens = len(tok.encode(raw))
char_tokens = len(raw)
print(f'char-level : {char_tokens:,} tokens')
print(f'bpe        : {bpe_tokens:,} tokens')
print(f'compression: {char_tokens / bpe_tokens:.2f}x')

LuffyTokenizer(vocab_size=2000)
char-level : 3,223,998 tokens
bpe        : 1,037,394 tokens
compression: 3.11x


In [12]:
# re-encode dataset with BPE
bpe_data = torch.tensor(tok.encode(raw), dtype=torch.long)
n = int(len(bpe_data) * 0.9)
train_data = bpe_data[:n]
val_data   = bpe_data[n:]

print(f'train: {len(train_data):,} tokens')
print(f'val  : {len(val_data):,} tokens')

train: 933,654 tokens
val  : 103,740 tokens


In [13]:
# override encode/decode to use BPE for train_generate_print
encode = tok.encode
decode = tok.decode

In [14]:
from gpt import GPT

bpe_model = GPT(
    tok.vocab_size,
    n_embd=384,
    context_size=256,
    n_head=6,
    n_layer=6
).to(device)

if torch.cuda.device_count() > 1:
    print(f'using {torch.cuda.device_count()} GPUs')
    bpe_model = nn.DataParallel(bpe_model)

print(f'params: {sum(p.numel() for p in bpe_model.parameters()) / 1e6:.2f}M')

train(bpe_model, steps=25000, batch_size=64, context_size=256, lr=3e-4)

start_idx = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = bpe_model.module if isinstance(bpe_model, nn.DataParallel) else bpe_model
with torch.no_grad():
    idx = start_idx
    for _ in range(300):
        idx_cond = idx[:, -256:]
        logits, _ = generated(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
print(tok.decode(idx[0].tolist()))

using 2 GPUs
params: 12.28M
step 0  train loss: 7.3452  val loss: 7.3360
step 1000  train loss: 3.5580  val loss: 3.9564
step 2000  train loss: 2.9354  val loss: 3.6527
step 3000  train loss: 2.5371  val loss: 3.6264
step 4000  train loss: 2.2180  val loss: 3.6926
step 5000  train loss: 1.9271  val loss: 3.8309
step 6000  train loss: 1.6846  val loss: 3.9753
step 7000  train loss: 1.4689  val loss: 4.1177
step 8000  train loss: 1.2982  val loss: 4.2732
step 9000  train loss: 1.1455  val loss: 4.3957
step 10000  train loss: 1.0143  val loss: 4.5511
step 11000  train loss: 0.9058  val loss: 4.6795
step 12000  train loss: 0.8118  val loss: 4.7885
step 13000  train loss: 0.7367  val loss: 4.8989
step 14000  train loss: 0.6593  val loss: 5.0085
step 15000  train loss: 0.6049  val loss: 5.1015
step 16000  train loss: 0.5450  val loss: 5.2216
step 17000  train loss: 0.5013  val loss: 5.3050
step 18000  train loss: 0.4617  val loss: 5.3816
step 19000  train loss: 0.4258  val loss: 5.4606
step 

In [15]:
model_to_save = bpe_model.module if isinstance(bpe_model, nn.DataParallel) else bpe_model
torch.save(model_to_save.state_dict(), 'luffy_gpt_bpe.pth')
print('saved → luffy_gpt_bpe.pth')

saved → luffy_gpt_bpe.pth
